# 5장 웹부록 | 작은 디퓨전 모델 돌려 보기

「영화로 보는 AI 리터러시」 5장 '그리는 AI: 이미지·영상 생성과 딥페이크'의 코랩 실습입니다.

손글씨 숫자 사진으로 작은 **디퓨전 모델**을 만들어 봅니다. 순수한 잡음에서 숫자가 한 단계씩 떠오르는 과정을 그림으로 보고, 단계 수를 줄이면 그림이 어떻게 달라지는지 비교합니다. 마지막으로 진짜 숫자 사진과 생성한 숫자 사진을 섞어 놓고 구별해 보는 **'사람 판별자' 놀이**를 합니다.

인터넷에서 큰 모델을 내려받지 않도록, 미리 학습된 모델을 불러오는 대신 **아주 작은 모델을 이 자리에서 30초~1분 동안 직접 학습**시킵니다.

**사용법**
- 위에서부터 차례로 각 칸 왼쪽의 ▶(실행) 버튼을 누르세요. 코드를 몰라도 괜찮습니다.
- `# ← 바꿔 보세요`라고 적힌 곳만 고쳐서 다시 ▶를 누르면 됩니다.
- 처음부터 다시 하고 싶으면 메뉴에서 **런타임 → 모두 실행**을 누르세요.
- 그래프 글자는 한글이 깨지지 않도록 영어로 적었습니다.

## 1단계. 손글씨 숫자 사진 불러오기

scikit-learn에 들어 있는 손글씨 숫자 1,797장을 씁니다. 사진 한 장은 가로세로 8칸, 모두 64개의 점입니다. 작지만 숫자는 알아볼 수 있습니다.

In [ ]:
import time, random
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
X = (digits.data / 8.0 - 1.0).astype(np.float32)   # 점의 밝기를 -1(흰색) ~ 1(검은색)로 맞춥니다
print(f"숫자 사진 {len(X):,}장, 한 장에 점 {X.shape[1]}개 (8×8)")

def show_row(images, titles=None, size=1.1, suptitle=None):
    fig, axes = plt.subplots(1, len(images), figsize=(len(images) * size, size * 1.3))
    for k, ax in enumerate(np.atleast_1d(axes)):
        ax.imshow(images[k].reshape(8, 8), cmap="gray_r", vmin=-1, vmax=1)
        ax.axis("off")
        if titles is not None:
            ax.set_title(titles[k], fontsize=9)
    if suptitle:
        plt.suptitle(suptitle)
    plt.show()

show_row(X[:12], [str(t) for t in digits.target[:12]], suptitle="Real handwritten digits")

## 2단계. 잡음 섞기: 쉬운 길

깨끗한 사진에 잡음을 조금씩 섞습니다(3-4절). `T`단계를 다 섞으면 사진은 완전히 잡음이 됩니다.
이 방향은 아무 배움 없이 계산만으로 됩니다.

In [ ]:
T = 50   # 잡음을 섞는 단계 수

# 단계마다 원래 그림이 얼마나 남아 있는지 정합니다 (처음엔 거의 다, 끝엔 거의 0)
s = 0.008
f = np.cos((np.arange(T + 1) / T + s) / (1 + s) * np.pi / 2) ** 2
keep = np.clip(f[1:] / f[0], 1e-4, 1.0).astype(np.float32)

def add_noise(x0, t, rng):
    noise = rng.standard_normal(x0.shape).astype(np.float32)
    return np.sqrt(keep[t]) * x0 + np.sqrt(1 - keep[t]) * noise

rng = np.random.default_rng(0)
steps_to_show = [0, 5, 10, 15, 20, 25, 30, 35, 40, 49]
show_row([add_noise(X[0], t, rng) for t in steps_to_show],
         [f"t={t}" for t in steps_to_show], suptitle="Adding noise step by step (the easy way)")
print("각 단계에서 원래 그림이 남은 비율:", [f"{keep[t]:.2f}" for t in steps_to_show])

## 3단계. 잡음 걷어 내기를 학습시키기: 어려운 길

이제 거꾸로 가는 길을 가르칩니다. 수많은 사진에 일부러 잡음을 섞어 놓고, 신경망에게 **잡음 섞인 사진과 단계 번호**를 보여 준 뒤 **잡음을 걷어 낸 깨끗한 사진**을 맞히게 합니다.
정답은 우리가 섞기 전의 사진이니 늘 알고 있습니다. 3장의 지도학습과 같습니다.

(책에서는 '섞인 잡음을 맞힌다'고 했습니다. 잡음 = 섞인 사진 − 깨끗한 사진이므로, 깨끗한 사진을 맞히는 것과 같은 일입니다. 작은 모델에서는 이쪽이 더 잘 됩니다.)

신경망은 3장에서 본 것과 같은 다층 신경망입니다. 30초~1분쯤 걸립니다.

👉 **바꿔 볼 곳**: `TRAIN_ROUNDS`를 줄이면 빨리 끝나지만 그림이 흐려집니다.

In [ ]:
import warnings
from sklearn.neural_network import MLPRegressor
warnings.filterwarnings("ignore")

COPIES = 12          # 사진 한 장에 잡음을 몇 가지로 섞어 연습시킬지
TRAIN_ROUNDS = 20    # ← 학습 바퀴 수

def time_code(t):
    """단계 번호를 신경망이 알아보기 쉬운 숫자 묶음으로 바꿉니다."""
    t = np.asarray(t, dtype=np.float32)[:, None] / T
    k = np.arange(1, 7, dtype=np.float32)[None, :]
    return np.concatenate([np.sin(np.pi * k * t / 2), np.cos(np.pi * k * t / 2)], axis=1)

rng = np.random.default_rng(0)
idx = np.repeat(np.arange(len(X)), COPIES)
t_train = rng.integers(0, T, len(idx))
x_clean = X[idx]
x_noisy = np.stack([add_noise(x_clean[i], t_train[i], rng) for i in range(len(idx))])
inputs = np.concatenate([x_noisy, time_code(t_train)], axis=1)

t0 = time.time()
denoiser = MLPRegressor(hidden_layer_sizes=(256, 256), max_iter=TRAIN_ROUNDS, batch_size=128,
                        learning_rate_init=1e-3, random_state=0)
denoiser.fit(inputs, x_clean)
print(f"연습 문제 {len(idx):,}개로 학습 끝! 걸린 시간 {time.time() - t0:.0f}초")

plt.figure(figsize=(6, 3))
plt.plot(denoiser.loss_curve_, "o-")
plt.xlabel("Training round")
plt.ylabel("Loss")
plt.title("The denoiser gets better")
plt.show()

## 4단계. 순수한 잡음에서 숫자 꺼내기

학습이 끝났습니다. 이제 **순수한 잡음**에서 출발해, 잡음을 조금씩 걷어 내는 일을 되풀이합니다.
한 단계마다 모델이 '깨끗한 그림은 아마 이럴 것'이라고 짐작하고, 그쪽으로 한 걸음 옮깁니다.

아래 그림의 줄 하나가 숫자 한 장이 떠오르는 과정입니다. 왼쪽 끝이 잡음, 오른쪽 끝이 완성된 그림입니다.

In [ ]:
def generate(n, steps=T, seed=1, record=False):
    """순수한 잡음 n장에서 출발해 steps번 잡음을 걷어 냅니다."""
    r = np.random.default_rng(seed)
    x = r.standard_normal((n, 64)).astype(np.float32)        # 순수한 잡음
    ts = np.linspace(T - 1, 0, steps).round().astype(int)    # 거쳐 갈 단계들
    history = [x.copy()]
    for i, t in enumerate(ts):
        guess = np.clip(denoiser.predict(np.concatenate([x, time_code(np.full(n, t))], axis=1)), -1, 1)
        if i == len(ts) - 1:
            x = guess
        else:
            t_next = ts[i + 1]
            noise_part = (x - np.sqrt(keep[t]) * guess) / np.sqrt(1 - keep[t])
            x = np.sqrt(keep[t_next]) * guess + np.sqrt(1 - keep[t_next]) * noise_part
        history.append(x.copy())
    return (x, history) if record else x

final, history = generate(4, steps=T, seed=7, record=True)
snap = [0, 5, 10, 20, 30, 40, 45, 48, 50]
fig, axes = plt.subplots(4, len(snap), figsize=(len(snap) * 1.1, 4 * 1.2))
for row in range(4):
    for col, k in enumerate(snap):
        axes[row, col].imshow(np.clip(history[k][row], -1, 1).reshape(8, 8), cmap="gray_r", vmin=-1, vmax=1)
        axes[row, col].axis("off")
        if row == 0:
            axes[row, col].set_title(f"step {k}", fontsize=9)
plt.suptitle("From pure noise to a digit (left -> right)")
plt.tight_layout()
plt.show()

처음 몇 단계에서 큰 모양(어디가 진하고 어디가 빈칸인지)이 정해지고, 뒤로 갈수록 획이 또렷해집니다. 책 그림 5-6의 '구도에서 세부로'와 같습니다.

출발한 잡음이 다르면 다른 숫자가 나옵니다. `SEED`를 바꿔 보세요.

In [ ]:
SEED = 1   # ← 숫자를 바꾸면 다른 잡음에서 출발합니다
samples = generate(16, steps=T, seed=SEED)
show_row(samples, suptitle=f"16 new digits made from noise (seed {SEED})", size=0.8)

## 5단계. 단계 수를 줄이면?

디퓨전은 여러 단계를 거치느라 그림 한 장에 시간이 걸립니다. 그래서 단계를 줄이는 연구가 활발합니다(3-4절 심화 상식).
**같은 잡음**에서 출발해 단계 수만 바꿔 봅니다.

👉 **바꿔 볼 곳**: `STEP_LIST`의 숫자를 바꿔 보세요 (1 이상 50 이하).

In [ ]:
STEP_LIST = [50, 20, 10, 5, 2, 1]   # ← 비교할 단계 수

fig, axes = plt.subplots(len(STEP_LIST), 10, figsize=(10 * 0.9, len(STEP_LIST) * 1.0))
for row, st in enumerate(STEP_LIST):
    t0 = time.time()
    imgs = generate(10, steps=st, seed=3)
    took = time.time() - t0
    for col in range(10):
        axes[row, col].imshow(imgs[col].reshape(8, 8), cmap="gray_r", vmin=-1, vmax=1)
        axes[row, col].axis("off")
    axes[row, 0].set_title(f"{st} steps ({took * 1000:.0f} ms)", fontsize=9, loc="left")
plt.tight_layout()
plt.show()

단계가 아주 적으면 빠르지만, 모델이 한 번에 '평균적인 숫자'를 짐작하게 되어 그림이 흐려지고 서로 비슷해집니다.

## 6단계. '사람 판별자' 놀이

3-2절의 GAN에서 판별자는 진짜와 가짜를 가려냈습니다. 이번에는 **여러분이 판별자**입니다.
아래 12장 가운데 6장은 진짜 손글씨, 6장은 방금 만든 그림입니다. 순서는 무작위로 섞었습니다.

In [ ]:
GAME_SEED = 0   # ← 숫자를 바꾸면 새 문제가 나옵니다

game_rng = random.Random(GAME_SEED)
real_idx = game_rng.sample(range(len(X)), 6)
fake = generate(6, steps=T, seed=100 + GAME_SEED)
pool = [("real", X[i]) for i in real_idx] + [("fake", f) for f in fake]
game_rng.shuffle(pool)

fig, axes = plt.subplots(2, 6, figsize=(9, 3.6))
for k, ax in enumerate(axes.ravel()):
    ax.imshow(pool[k][1].reshape(8, 8), cmap="gray_r", vmin=-1, vmax=1)
    ax.set_title(f"#{k + 1}")
    ax.axis("off")
plt.suptitle("Which ones are made by AI?")
plt.tight_layout()
plt.show()

👉 **바꿔 볼 곳**: AI가 만든 것 같은 그림의 번호를 `MY_GUESS`에 적고 실행하세요. 예: `MY_GUESS = [2, 5, 7, 8, 10, 12]`
아무것도 적지 않고 실행하면 정답만 보여 줍니다.

In [ ]:
MY_GUESS = []   # ← AI가 만든 것 같은 번호 6개를 적어 보세요

answer = [k + 1 for k, (kind, _) in enumerate(pool) if kind == "fake"]
print("AI가 만든 그림:", answer)
if MY_GUESS:
    right = sum(1 for k in range(1, 13) if (k in MY_GUESS) == (k in answer))
    print(f"12장 가운데 {right}장을 바르게 가려냈습니다. (무작위로 찍으면 평균 6장)")
    if right <= 8:
        print("가려내기 어려웠다면, 이 작은 모델도 이미 사람 판별자를 꽤 속였다는 뜻입니다.")
else:
    print("MY_GUESS에 번호를 적고 다시 실행하면 점수를 알려 줍니다.")

### 더 해 보기: 모델은 학습 사진을 베꼈을까?

생성한 그림마다 학습 사진 1,797장 가운데 **가장 닮은 사진**을 찾아 나란히 놓아 봅니다. 아주 똑같다면 '새 그림'이 아니라 베낀 것에 가깝습니다. 4-2절의 'AI가 만든 그림은 누구의 것인가'와 이어지는 질문입니다.

In [ ]:
made = generate(8, steps=T, seed=11)
dist = ((made[:, None, :] - X[None, :, :]) ** 2).mean(axis=2)
closest = dist.argmin(axis=1)

fig, axes = plt.subplots(2, 8, figsize=(8 * 1.0, 2.6))
for k in range(8):
    axes[0, k].imshow(made[k].reshape(8, 8), cmap="gray_r", vmin=-1, vmax=1)
    axes[1, k].imshow(X[closest[k]].reshape(8, 8), cmap="gray_r", vmin=-1, vmax=1)
    axes[1, k].set_title(f"diff {dist[k, closest[k]]:.2f}", fontsize=8)
    axes[0, k].axis("off"); axes[1, k].axis("off")
axes[0, 0].set_title("generated", fontsize=9, loc="left")
plt.suptitle("Top: generated / Bottom: most similar training image")
plt.tight_layout()
plt.show()

## 생각해 볼 것

1. 모델은 '숫자 7은 이렇게 생겼다'는 규칙을 배운 적이 없습니다. 그런데도 잡음에서 숫자가 떠오르는 이유는 무엇일까요? 3-4절 심화 상식의 '배운 그럴듯한 모양'과 연결해 설명해 보세요.
2. 사람 판별자 놀이에서 몇 장을 맞혔습니까? 8×8 숫자도 가리기 어렵다면, 고해상도 얼굴 사진의 딥페이크는 어떨까요? 3-8절의 세 가지 방법(탐지, 워터마크, 출처 기록) 가운데 무엇이 가장 믿을 만할까요?
3. 생성한 그림이 학습 사진과 거의 같다면, 그 그림은 누구의 것일까요? 4-2절의 저작권 다툼과 비교해 보세요.